# Практика PyTorch

In [1]:
import time
import numpy as np
import torch

print(torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

2.14.0+cu130
cuda NVIDIA GeForce RTX 5060 Ti


# Урок 1. Тензоры и работа с GPU

## 1. Что такое тензор

**Тензор** (`torch.Tensor`) — многомерный массив чисел одного типа. Он похож на `np.ndarray`, но у него есть два важных отличия:

1. тензор может лежать на GPU и считаться там;
2. тензор может участвовать в автоматическом дифференцировании (`autograd`, урок 2).

У тензора есть три основных атрибута:

| Атрибут | Что хранит | Пример |
|---|---|---|
| `x.shape` | размеры по каждой оси | `torch.Size([32, 3, 224, 224])` |
| `x.dtype` | тип элементов | `torch.float32` |
| `x.device` | где лежит тензор | `cpu`, `cuda:0` |

### Создание тензоров

```python
torch.tensor([[1., 2.], [3., 4.]])   # из данных (данные КОПИРУЮТСЯ)
torch.zeros(3, 4)                     # нули
torch.ones(3, 4)                      # единицы
torch.arange(0, 10, 2)                # 0, 2, 4, 6, 8
torch.linspace(0, 1, 5)               # 5 равноотстоящих точек
torch.eye(3)                          # единичная матрица
torch.rand(3, 4)                      # равномерно на [0, 1)
torch.randn(3, 4)                     # нормальное N(0, 1)
torch.zeros_like(x)                   # нули с формой/dtype/device как у x
```

### Связь с NumPy

```python
a = np.arange(6, dtype=np.float32)
t = torch.from_numpy(a)   # t и a ДЕЛЯТ одну память
t[0] = 100                # изменится и a[0]
b = t.numpy()             # обратно, тоже без копирования (только на CPU)
```

`torch.tensor(a)` создаёт копию, а `torch.from_numpy(a)` — нет. Это частый источник ошибок.

## 2. dtype

По умолчанию:
- числа с точкой → `float32`;
- целые числа → `int64`.

Основные типы: `float32` (стандарт для обучения), `float16`/`bfloat16` (ускорение на GPU, урок 7), `float64` (высокая точность), `int64` (индексы, метки классов), `bool` (маски).

```python
x = torch.tensor([1, 2, 3])        # int64
y = x.float()                      # float32
z = x.to(torch.float16)            # явное приведение
```

Арифметика над тензорами разных dtype подчиняется правилам **type promotion**: результат получает более «широкий» тип. Метки классов для `CrossEntropyLoss` должны быть `int64`, а не `float`.

## 3. device и GPU

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x = torch.randn(1000, 1000)          # на CPU
x_gpu = x.to(device)                 # копия на GPU
y_gpu = torch.randn(1000, 1000, device=device)  # сразу создаём на GPU
x_cpu = x_gpu.cpu()                  # обратно на CPU
```

Правила:
- операция возможна только между тензорами **на одном устройстве**; иначе будет `RuntimeError`;
- копирование между CPU и GPU идёт по шине PCIe и стоит дорого, поэтому данные и модель стараются держать на GPU целиком;
- `x.numpy()` работает только для CPU-тензоров (для GPU сначала `.cpu()`);
- `x.item()` достаёт число из тензора с одним элементом (и заставляет GPU дождаться вычислений).

### Асинхронность GPU и измерение времени

Вызовы CUDA **асинхронны**: Python ставит операцию в очередь и идёт дальше, не дожидаясь результата. Если замерить время обычным `time.time()` без синхронизации, получится время постановки в очередь, а не время вычисления. Правильно так:

```python
torch.cuda.synchronize()
t0 = time.perf_counter()
y = x @ x
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0
```

Первый запуск на GPU включает инициализацию (прогрев), поэтому перед замером делают 1–2 «разогревочных» вызова.

## 4. Индексация и срезы

Работает так же, как в NumPy:

```python
x = torch.arange(24).reshape(4, 6)
x[0]            # первая строка
x[:, 1]         # второй столбец
x[1:3, ::2]     # срезы с шагом
x[x > 10]       # булева маска -> одномерный тензор
x[[0, 2], [1, 3]]   # целочисленная индексация: элементы (0,1) и (2,3)
x[..., 0]       # Ellipsis: последняя ось = 0, остальные целиком
x[:, None, :]   # добавить ось (None == unsqueeze)
```

Срезы (`x[1:3]`) возвращают **view** (общая память), а индексация списком или маской — **копию**.

## 5. view, reshape, permute: что происходит в памяти

Тензор хранится в памяти как плоский одномерный массив, а форма задаётся метаданными: `shape` и `stride` (на сколько элементов надо шагнуть в памяти, чтобы сдвинуться на 1 по каждой оси).

```python
x = torch.arange(6).reshape(2, 3)
x.stride()          # (3, 1)
x.t().stride()      # (1, 3)  — транспонирование не двигает данные, меняет только stride
x.t().is_contiguous()   # False
```

| Операция | Копирует данные? | Замечание |
|---|---|---|
| `view(...)` | нет | требует совместимых strides, иначе ошибка |
| `reshape(...)` | если нужно | возвращает view, когда возможно, иначе копию |
| `permute`, `transpose`, `t()` | нет | результат обычно не contiguous |
| `contiguous()` | если нужно | делает копию с «правильным» порядком в памяти |
| `squeeze`, `unsqueeze`, `expand` | нет | `expand` не выделяет память |
| `clone()` | да | явная копия |

Практическое правило: после `permute`/`transpose` `view` часто падает, тогда нужен `reshape` или `.contiguous().view(...)`.

Операции с суффиксом `_` (`x.add_(1)`, `x.zero_()`) работают **in-place**. Они изменяют тензор на месте, а значит и все его view. С autograd in-place операции нужно использовать осторожно (урок 2).

## 6. Broadcasting

Когда формы операндов разные, PyTorch выравнивает их **справа налево**. Две размерности совместимы, если они равны, или одна из них равна 1, или её нет. Размерность 1 «растягивается» без копирования памяти.

```
A:     (4, 1, 5)
B:        (3, 5)
A + B: (4, 3, 5)
```

```python
a = torch.randn(4, 1, 5)
b = torch.randn(3, 5)
(a + b).shape    # torch.Size([4, 3, 5])
```

Типичная ловушка:

```python
a = torch.randn(5)
b = torch.randn(5, 1)
(a + b).shape    # (5, 5), а не (5,): вектор и столбец дают матрицу
```

Типичный приём — превратить вычисления в цикле по парам в одну операцию над тензором с добавленной осью:

```python
A = torch.randn(100, 3)   # N точек
B = torch.randn(200, 3)   # M точек
diff = A[:, None, :] - B[None, :, :]   # (100, 200, 3)
dist = diff.pow(2).sum(-1).sqrt()      # (100, 200)
```

## 7. Редукции и `einsum`

Редукции (`sum`, `mean`, `max`, `min`, `argmax`, `std`, `norm`) принимают `dim` (ось) и `keepdim`:

```python
x = torch.randn(4, 5)
x.sum(dim=1).shape                 # (4,)
x.sum(dim=1, keepdim=True).shape   # (4, 1)  — удобно для последующего broadcasting
```

`torch.einsum` описывает операцию над осями строкой:

```python
torch.einsum("ij,jk->ik", A, B)         # матричное умножение
torch.einsum("bij,bjk->bik", A, B)      # батчевое матричное умножение
torch.einsum("bi,bi->b", u, v)          # скалярные произведения по батчу
```

Индексы, которых нет справа от `->`, суммируются. Для матричных умножений есть `@` (`torch.matmul`) и `torch.bmm`; они обычно быстрее и читабельнее на простых случаях.

## 8. Числовая устойчивость

Формулы на бумаге и в коде ведут себя по-разному. Пример — **softmax**:

$$\mathrm{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Для больших $x_i$ значение $e^{x_i}$ переполняется до `inf`, и получается `nan`. Решение — вычесть максимум (результат не изменится, так как числитель и знаменатель домножаются на одну и ту же константу):

$$\mathrm{softmax}(x)_i = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}, \quad m = \max_j x_j$$

Похожая проблема при вычислении расстояний через формулу $\|a-b\|^2 = \|a\|^2 + \|b\|^2 - 2\,a\cdot b$: из-за ошибок округления результат может получиться чуть меньше нуля, и `sqrt` даст `nan`. Перед корнем значения обрезают снизу нулём (`clamp(min=0)`).

## Задачи урока 1

### Задача 1.1. Попарные расстояния тремя способами

Даны `A` формы `(N, D)` и `B` формы `(M, D)`. Нужно получить матрицу `(N, M)` евклидовых расстояний.

Реализуйте три функции:
1. `dist_loops(A, B)`: два вложенных цикла по строкам;
2. `dist_broadcast(A, B)`: через broadcasting с добавлением осей;
3. `dist_matmul(A, B)`: через формулу $\|a\|^2 + \|b\|^2 - 2ab^\top$ и матричное умножение. Защититесь от отрицательных значений под корнем.

Проверьте все три против `torch.cdist(A, B)`. Затем для `N = M = 2000`, `D = 128` на CPU и на GPU замерьте время (с `synchronize` и прогревом) и объясните разницу между вариантами 2 и 3 (подсказка: посмотрите, какой промежуточный тензор создаёт каждый из них).

### Задача 1.2. Устойчивый softmax

Реализуйте `softmax(x, dim)` вручную с вычитанием максимума. Покажите на тензоре с большими значениями (например, `x = torch.tensor([[1000., 1001., 1002.]])`), что наивная версия `exp(x) / exp(x).sum(...)` выдаёт `nan`, а ваша совпадает с `torch.softmax`. Проверьте, что работает для произвольной оси `dim`, в том числе отрицательной.

### Задача 1.3. kNN в батче без циклов

Дан тензор `X` формы `(B, N, D)`: `B` облаков по `N` точек. Для каждой точки найдите `K` ближайших соседей **внутри её облака** (саму точку не считать соседом) и верните:
- индексы соседей `idx` формы `(B, N, K)`;
- координаты соседей `nbrs` формы `(B, N, K, D)`.

Циклов по батчу и по точкам быть не должно. Подсказки: батчевая матрица расстояний `(B, N, N)`, `torch.topk(..., largest=False)`, для сбора координат — продвинутая индексация или `torch.gather`. Проверьте результат на небольшом примере наивной реализацией с циклами.

### Задача 1.4. Когда GPU выигрывает

Для квадратных матриц размеров `n ∈ {64, 256, 1024, 4096}` замерьте время `A @ B` на CPU и на GPU (по 10 повторов после прогрева, правильно синхронизируя GPU). Составьте таблицу и ответьте письменно:
- при каком `n` GPU становится быстрее и почему на малых размерах он может проигрывать;
- что изменится в замерах, если включить в них время копирования `A` и `B` на GPU.

### Задача 1.5. Что изменится в памяти (предсказать, потом проверить)

Для каждого фрагмента сначала запишите ответ в комментарии, потом запустите и сравните.

```python
# (а)
x = torch.arange(12).reshape(3, 4)
y = x.t()
y[0, 0] = 100
# Чему равно x[0, 0]? Почему?

# (б)
x = torch.arange(12).reshape(3, 4)
y = x.t().view(12)
# Что произойдёт? Как исправить двумя способами и чем они различаются?

# (в)
x = torch.arange(12).reshape(3, 4)
y = x[x > 5]
y[0] = -1
# Изменится ли x? Почему?

# (г)
a = torch.randn(5)
b = torch.randn(5, 1)
c = a * b
# Какая форма у c? Какую ошибку чаще всего допускают с таким кодом?
```

# Урок 2. Autograd

## 1. Идея

В уроке 1 («Цепное правило и вычислительный граф») вы вручную писали `f_forward` и `f_backward`. **Autograd** делает это автоматически: пока вы считаете прямой проход, PyTorch записывает операции в граф, а при вызове `backward()` проходит по этому графу в обратном порядке и применяет цепное правило.

Это **reverse-mode автоматическое дифференцирование** (backprop). Для функции $f:\mathbb{R}^n \to \mathbb{R}$ он считает весь градиент за стоимость, сопоставимую с одним прямым проходом, независимо от $n$. Поэтому он подходит для нейросетей, где параметров миллионы, а лосс один.

## 2. Основы: `requires_grad`, `backward`, `.grad`

```python
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3 + 2 * x      # y = x^3 + 2x
y.backward()            # считает dy/dx
x.grad                  # tensor(14.)  (3*x^2 + 2 = 14)
```

- `requires_grad=True` включает отслеживание операций для этого тензора. Все тензоры, полученные из него, тоже будут отслеживаться.
- `backward()` без аргументов можно вызывать только на **скаляре** (тензоре из одного элемента).
- Градиент записывается в `x.grad` только для **листовых** тензоров (созданных пользователем, а не как результат операции) с `requires_grad=True`.
- Параметры слоёв `nn.Module` — листовые тензоры с `requires_grad=True`. Именно их потом обновляет оптимизатор.

## 3. Граф вычислений: `grad_fn`

Каждый тензор, полученный в результате операции над отслеживаемыми тензорами, хранит ссылку `grad_fn` на узел графа, который его создал:

```python
a = torch.tensor(3.0, requires_grad=True)
b = a * 2
c = b ** 2
print(a.is_leaf, b.is_leaf)     # True False
print(b.grad_fn)                # <MulBackward0 ...>
print(c.grad_fn.next_functions) # ссылки на предыдущие узлы
```

Граф **динамический** (define-by-run): он строится заново при каждом прямом проходе. Поэтому в forward можно использовать обычные `if`, циклы и произвольный Python-код.

После `backward()` граф по умолчанию **освобождается** (буферы, нужные для обратного прохода, удаляются). Второй вызов `backward()` по тому же графу даст ошибку, если не передать `retain_graph=True`.

## 4. Накопление градиентов

`backward()` **прибавляет** градиент к `.grad`, а не перезаписывает его:

```python
x = torch.tensor(1.0, requires_grad=True)
(x * 3).backward()
(x * 3).backward()
x.grad        # tensor(6.), а не 3
x.grad.zero_()  # обнулить
```

Поэтому в цикле обучения градиенты обнуляют перед каждым шагом (`optimizer.zero_grad()`). Накопление при этом полезно: оно позволяет складывать градиенты по нескольким микробатчам (gradient accumulation).

Ещё одно следствие: если тензор используется в графе несколько раз, градиенты по всем путям **суммируются**. Это прямое следствие цепного правила для функций нескольких аргументов.

## 5. Векторный результат и якобиан

Для невекторного `y` (например, формы `(3,)`) `y.backward()` без аргумента не сработает: непонятно, какую скалярную величину дифференцировать. Нужно передать вектор `v` той же формы, что и `y`:

```python
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2
y.backward(torch.ones_like(y))   # то же, что y.sum().backward()
x.grad                            # tensor([2., 4., 6.])
```

Математически `backward(v)` считает **произведение вектора на якобиан** (vector-Jacobian product):

$$x.\text{grad} = v^\top J, \qquad J_{ij} = \frac{\partial y_i}{\partial x_j}$$

Полный якобиан целиком autograd не строит, он нужен редко. При необходимости его даёт `torch.autograd.functional.jacobian`.

## 6. Управление отслеживанием

| Инструмент | Что делает |
|---|---|
| `with torch.no_grad():` | внутри блока операции не записываются в граф; экономит память, используется при инференсе и валидации |
| `torch.inference_mode()` | как `no_grad`, но ещё быстрее; тензоры, созданные внутри, нельзя потом использовать в графе |
| `x.detach()` | тензор с теми же данными, но **отрезанный от графа** (общая память с `x`) |
| `x.requires_grad_(False)` | отключает отслеживание «на месте»; так «замораживают» слои |

```python
x = torch.tensor(2.0, requires_grad=True)
y = (x * 3).detach()   # y не знает, что произошло из x
z = y * x
z.backward()
x.grad                  # tensor(6.): градиент идёт только через второй множитель, через y — нет
```

## 7. In-place операции и ошибки версий

Для некоторых операций обратный проход использует значения, сохранённые при прямом. Если после этого изменить такой тензор на месте, градиент был бы неверным. PyTorch считает версии тензоров и бросает ошибку:

```python
a = torch.tensor([1., 2.], requires_grad=True)
b = a.exp()
b.add_(1)          # изменили результат exp
b.sum().backward() # RuntimeError: ... modified by an inplace operation
```

Правило: на тензорах, участвующих в графе, избегайте in-place операций, если не уверены. Листовой тензор с `requires_grad=True` нельзя менять in-place вне `no_grad` (ошибка). Параметры оптимизатор обновляет как раз внутри `no_grad`.

## 8. Градиенты высших порядков

`torch.autograd.grad` возвращает градиенты как обычные значения, не трогая `.grad`. С `create_graph=True` сам обратный проход тоже записывается в граф, и его можно дифференцировать ещё раз:

```python
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
(g,) = torch.autograd.grad(y, x, create_graph=True)   # 3x^2 = 12
(g2,) = torch.autograd.grad(g, x)                      # 6x   = 12
```

Это нужно, например, для штрафов на градиент (gradient penalty), MAML, вычисления гессиана.

## 9. Собственные операции: `torch.autograd.Function`

Если операции нет в PyTorch, она численно неустойчива или вы хотите задать нестандартный градиент, описывают её вручную: пишут `forward` и `backward` как статические методы.

```python
class MySquare(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)      # сохраняем то, что понадобится в backward
        return x ** 2

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x    # цепное правило: dL/dx = dL/dy * dy/dx

y = MySquare.apply(torch.tensor(3.0, requires_grad=True))
```

Что важно:
- `backward` получает `grad_output` (градиент лосса по выходу) и должен вернуть градиент по **каждому** входу `forward` (для нетензорных входов — `None`);
- тензоры, нужные в `backward`, сохраняют через `ctx.save_for_backward`, а не через атрибуты `ctx`;
- внутри `forward` отслеживание градиентов отключено.

### Проверка градиента: `gradcheck`

Как и в уроке 1 с `num_grad`, аналитический градиент проверяют сравнением с численным:

```python
from torch.autograd import gradcheck
x = torch.randn(5, dtype=torch.double, requires_grad=True)   # ВАЖНО: float64
gradcheck(MySquare.apply, (x,), eps=1e-6, atol=1e-4)          # True или исключение
```

`gradcheck` надёжно работает только в `float64`: в `float32` ошибки округления сравнимы с шагом конечных разностей.

## 10. Типичные ошибки

1. Забыли `zero_grad()` → градиенты копятся между шагами.
2. Повторный `backward()` без `retain_graph=True` → ошибка про освобождённый граф.
3. `loss.backward()` на нескалярном лоссе без аргумента.
4. Копирование тензора через `.numpy()` или `float()` внутри графа обрывает отслеживание; для накопления метрик используют `loss.item()` или `loss.detach()`.
5. Накопление `total_loss += loss` (вместо `loss.item()`) тянет за собой весь граф и расходует память.
6. In-place операции на нужных для backward тензорах.
7. Валидация без `no_grad`: впустую расходуется память.

## Задачи урока 2

Сначала предсказывайте результат в комментариях, потом запускайте.

### Задача 2.1. Сравнение с ручным backward

Возьмите функцию и свой `f_forward`/`f_backward` из урока 1 (задача 1.1) либо любую функцию вида
$f(x, w) = \mathrm{Huber}\big(\sigma(x w) - y\big)$ со скалярами/векторами по вашему выбору. Вычислите градиенты по всем параметрам тремя способами:
1. вручную (по цепному правилу);
2. через `autograd` (`backward()` или `torch.autograd.grad`);
3. численно (`num_grad` из урока 1).

Сравните результаты через `torch.allclose` и выведите максимальную абсолютную разницу между каждой парой способов. Объясните, почему расхождение между autograd и численным градиентом больше, чем между autograd и ручным.

### Задача 2.2. Исследование графа

Для выражения
```python
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
c = a * b
d = c + a
e = d ** 2
```
- выведите `grad_fn` и `is_leaf` для каждого из пяти тензоров;
- напишите функцию `print_graph(t)`, которая рекурсивно обходит `t.grad_fn.next_functions` и печатает дерево узлов с отступами (листовые тензоры показываются как `AccumulateGrad`);
- вручную вычислите `de/da` и `de/db` и проверьте с `backward()`. Обратите внимание, что `a` участвует в графе двумя путями. Покажите в комментарии, какие два слагаемых складываются.

### Задача 2.3. Накопление градиентов

Есть набор данных `X` формы `(64, 10)`, `y` формы `(64,)` и линейная модель `w` формы `(10,)` (листовой тензор). Лосс — MSE.
1. Посчитайте градиент по всему батчу за один `backward()`.
2. Разбейте батч на 4 микробатча по 16 и накопите градиент по ним через повторные `backward()`. Подберите масштаб лосса на микробатче так, чтобы итоговый `w.grad` совпал с пунктом 1 (`torch.allclose`).
3. Покажите, что получится, если масштаб не учитывать, и объясните почему.

### Задача 2.4. Собственная операция

Реализуйте `torch.autograd.Function` для функции **clipped ReLU** с обучаемой верхней границей не нужно, достаточно фиксированной: $f(x) = \min(\max(x, 0), c)$, где $c$ передаётся как число (нетензорный аргумент).
- Градиент определите так: 1 внутри $(0, c)$, иначе 0.
- Сохраните в `forward` только то, что нужно для `backward` (подумайте, достаточно ли сохранить булеву маску вместо `x`).
- Проверьте через `gradcheck` в `float64` на случайном входе. Выберите вход так, чтобы значения не оказывались вблизи точек излома 0 и $c$, и объясните, почему `gradcheck` в этих точках ненадёжен.
- Сравните результат с встроенным `torch.clamp(x, 0, c)` по значениям и по градиентам.

### Задача 2.5. Straight-through estimator

Функция округления `torch.round` имеет нулевой градиент почти везде, поэтому через неё нельзя обучать. Реализуйте через `autograd.Function` операцию `round_ste`:
- в прямом проходе округляет до ближайшего целого;
- в обратном проходе пропускает градиент «как есть» (как у тождественной функции).

Затем найдите с её помощью $w$, при котором $\mathrm{round\_ste}(w \cdot x) = 7$ для $x = 2.5$, минимизируя $(\mathrm{round\_ste}(w x) - 7)^2$ градиентным спуском с `lr=0.01` и 100 шагами. Покажите траекторию $w$ и результат. Сравните с версией на обычном `torch.round` и объясните, почему та не обучается.

### Задача 2.6. Градиент второго порядка

Для $f(x) = x^3 \sin(x)$ найдите $f'(x)$ и $f''(x)$ в точке $x = 1.5$ через `torch.autograd.grad` с `create_graph=True`. Проверьте аналитически (выведите формулы на бумаге). Затем для функции двух переменных $g(x, y) = x^2 y + y^3$ вычислите **гессиан** (матрицу вторых производных 2×2) в точке $(1, 2)$ вручную через циклы `autograd.grad` по компонентам градиента и сравните с `torch.autograd.functional.hessian`.

### Задача 2.7. Поиск ошибок (разбор кода)

Объясните, что и почему идёт не так в каждом фрагменте, и исправьте:

```python
# (а)
w = torch.randn(3, requires_grad=True)
for _ in range(3):
    loss = (w ** 2).sum()
    loss.backward()
    w -= 0.1 * w.grad
# Ошибка выполнения. Какая и почему? Как исправить?

# (б)
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x.detach().exp()
y.sum().backward()
# Что произойдёт?

# (в)
x = torch.tensor(1.0, requires_grad=True)
y = x * 2
y.backward()
y.backward()
# Что произойдёт и как сделать так, чтобы второй вызов сработал?

# (г)
losses = []
for step in range(1000):
    loss = model(x).pow(2).mean()
    loss.backward()
    losses.append(loss)
# Что не так с расходом памяти?
```